# Week 1 &mdash; Geometric Priors and Symmetry

## Theory: Manifolds, Equivariance, and Symmetry Groups

**Geometric Deep Learning &mdash; From Foundations to Applications**

---

### Learning Objectives

By the end of this notebook, you should be able to:

1. Articulate why high-dimensional learning is intractable without *structural priors*.
2. Define the central objects of geometric deep learning: **domains**, **signals**, and **symmetry groups**.
3. Distinguish rigorously between **invariance** and **equivariance**.
4. Describe the **Lie groups** $SO(3)$ and $SE(3)$ and their action on Euclidean space.
5. Recognise how the *Geometric Deep Learning blueprint* unifies CNNs, GNNs, and transformers under a single principle.

---


## 1. The Curse of Dimensionality and the Need for Priors

Consider the canonical supervised learning problem: we seek a function
$$
f^\star : \mathcal{X} \to \mathcal{Y}
$$
from a (possibly high-dimensional) input space $\mathcal{X} \subseteq \mathbb{R}^d$ to a label space $\mathcal{Y}$, given a finite sample $\{(x_i, y_i)\}_{i=1}^n$.

If we assume only that $f^\star$ is Lipschitz-continuous, then approximating it to accuracy $\varepsilon$ in the uniform norm requires on the order of
$$
n = \mathcal{O}\!\left(\varepsilon^{-d}\right)
$$
samples. This *exponential* dependence on the ambient dimension $d$ is the **curse of dimensionality**. For images at modest resolution ($d \sim 10^5$) or molecular point clouds ($d \sim 10^3$), uniform approximation is hopeless.

The resolution is not to learn over arbitrary functions, but to restrict the hypothesis class using **priors** that reflect the *structure of the data domain*. Geometric Deep Learning (GDL) makes one prior central:

> **Geometric prior.** The data lives on a domain $\Omega$ equipped with a symmetry group $\mathfrak{G}$, and the target function $f^\star$ respects that symmetry.

This single assumption &mdash; symmetry &mdash; dramatically shrinks the effective hypothesis space and underlies the sample efficiency of convolutional networks, graph networks, and equivariant architectures alike.


## 2. Domains, Signals, and Group Actions

We formalise the setting following Bronstein et al. (2021).

**Domain.** Let $\Omega$ be a set (the *domain*), e.g. a grid $\mathbb{Z}^2$, a graph $G = (V, E)$, a sphere $S^2$, or a Riemannian manifold $\mathcal{M}$.

**Signals.** A *signal* on $\Omega$ with values in a vector space $\mathcal{C}$ is a function $x : \Omega \to \mathcal{C}$. The space of such signals,
$$
\mathcal{X}(\Omega, \mathcal{C}) = \{\, x : \Omega \to \mathcal{C} \,\},
$$
is itself a vector space under pointwise operations.

**Symmetry group.** A *symmetry* of $\Omega$ is a transformation that preserves its structure. The collection of all such transformations forms a **group** $\mathfrak{G}$: a set with an associative composition, an identity $e$, and inverses.

**Group action.** The group acts on signals by *pullback*. For $\mathfrak{g} \in \mathfrak{G}$ acting on the domain via $\mathfrak{g} : \Omega \to \Omega$, the induced action on a signal $x$ is
$$
(\rho(\mathfrak{g})\, x)(u) = x\big(\mathfrak{g}^{-1} u\big), \qquad u \in \Omega.
$$
The map $\rho : \mathfrak{G} \to \mathrm{GL}(\mathcal{X})$ is a **group representation**: it satisfies $\rho(\mathfrak{g}\mathfrak{h}) = \rho(\mathfrak{g})\rho(\mathfrak{h})$.


## 3. Invariance and Equivariance

These two notions are the conceptual heart of the course. Let $f : \mathcal{X} \to \mathcal{Y}$ be a map, and suppose $\mathfrak{G}$ acts on $\mathcal{X}$ via $\rho$ and on $\mathcal{Y}$ via $\sigma$.

**Definition (Invariance).** $f$ is *$\mathfrak{G}$-invariant* if
$$
f(\rho(\mathfrak{g})\, x) = f(x) \qquad \text{for all } \mathfrak{g} \in \mathfrak{G},\; x \in \mathcal{X}.
$$
The output does not change when the input is transformed. *Classification of an object's category is invariant to its position.*

**Definition (Equivariance).** $f$ is *$\mathfrak{G}$-equivariant* if
$$
f(\rho(\mathfrak{g})\, x) = \sigma(\mathfrak{g})\, f(x) \qquad \text{for all } \mathfrak{g} \in \mathfrak{G},\; x \in \mathcal{X}.
$$
Transforming the input transforms the output *correspondingly*. *Semantic segmentation is equivariant: translate the image, and the segmentation map translates with it.*

Invariance is the special case of equivariance in which $\sigma(\mathfrak{g}) = \mathrm{id}$ for all $\mathfrak{g}$. Deep equivariant networks typically maintain equivariance through their hidden layers and impose invariance only at the final read-out, preserving as much geometric information as possible internally.


In [ ]:
import numpy as np

# A concrete illustration of equivariance with the 90-degree rotation group C4
# acting on a small 2D image (signal on a grid).

def rot90(img, k=1):
    """Rotate an image by k * 90 degrees counter-clockwise."""
    return np.rot90(img, k)

# A toy 'edge detector' implemented as a rotation-equivariant operation:
# we compute the magnitude of the discrete gradient, which is invariant to
# rotation, while the gradient field itself is equivariant.

def gradient_magnitude(img):
    gy, gx = np.gradient(img.astype(float))
    return np.sqrt(gx**2 + gy**2)

rng = np.random.default_rng(0)
x = rng.integers(0, 10, size=(4, 4)).astype(float)

# Equivariance check for the SCALAR field 'gradient magnitude':
# rotating the input then filtering == filtering then rotating.
lhs = gradient_magnitude(rot90(x))
rhs = rot90(gradient_magnitude(x))

print("Input image:\n", x)
print("\nMax |LHS - RHS| for gradient magnitude under 90-deg rotation:",
      np.max(np.abs(lhs - rhs)))


The near-zero discrepancy above (up to discretisation effects at the boundary) confirms that the gradient-magnitude operator commutes with the rotation action: it is an *equivariant* feature map. This is precisely the property we will engineer into network layers.


## 4. Lie Groups: $SO(3)$ and $SE(3)$

Many physically relevant symmetries are *continuous*. The appropriate language is that of **Lie groups** &mdash; groups that are also smooth manifolds, so that we may differentiate the group action.

### 4.1 The special orthogonal group $SO(3)$

$SO(3)$ is the group of **rotations** of $\mathbb{R}^3$ about the origin:
$$
SO(3) = \{\, R \in \mathbb{R}^{3\times 3} : R^\top R = I,\; \det R = +1 \,\}.
$$
The constraint $R^\top R = I$ preserves the Euclidean inner product, hence lengths and angles; $\det R = +1$ excludes reflections. $SO(3)$ is a compact, three-dimensional, *non-abelian* Lie group: rotations about different axes do not commute.

### 4.2 The special Euclidean group $SE(3)$

$SE(3)$ describes **rigid-body motions** &mdash; rotations *and* translations:
$$
SE(3) = \big\{\, (R, t) : R \in SO(3),\; t \in \mathbb{R}^3 \,\big\}, \qquad (R,t)\cdot x = Rx + t.
$$
Composition is the semidirect product $(R_1,t_1)(R_2,t_2) = (R_1 R_2,\; R_1 t_2 + t_1)$, often written $SE(3) = SO(3) \ltimes \mathbb{R}^3$. This is the symmetry group of three-dimensional physical space, and therefore the relevant group for molecular structures, point clouds, and robotics &mdash; topics we revisit in Week 4.


In [ ]:
# Verifying group axioms numerically for SO(3): closure and orthogonality.
from scipy.spatial.transform import Rotation as Rot

# Two random rotations.
R1 = Rot.random(random_state=1).as_matrix()
R2 = Rot.random(random_state=2).as_matrix()

# Orthogonality and unit determinant.
print("R1^T R1 = I?   ", np.allclose(R1.T @ R1, np.eye(3)))
print("det(R1) = +1?  ", np.isclose(np.linalg.det(R1), 1.0))

# Closure: the product of two rotations is again a rotation.
R12 = R1 @ R2
print("R1 R2 in SO(3)?", np.allclose(R12.T @ R12, np.eye(3))
      and np.isclose(np.linalg.det(R12), 1.0))

# Non-commutativity (SO(3) is non-abelian).
print("R1 R2 == R2 R1?", np.allclose(R1 @ R2, R2 @ R1))


In [ ]:
# An SE(3) transformation as a 4x4 homogeneous matrix, demonstrating
# the action g . x = R x + t on a point cloud.

def se3(R, t):
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

R = Rot.from_euler('z', 90, degrees=True).as_matrix()
t = np.array([1.0, 2.0, 0.0])
T = se3(R, t)

# A small point cloud (homogeneous coordinates).
pts = np.array([[1, 0, 0, 1],
                [0, 1, 0, 1],
                [0, 0, 1, 1]], dtype=float).T   # shape (4, N)

transformed = (T @ pts)[:3].T
print("Transformed points (R x + t):\n", np.round(transformed, 3))


## 5. The Geometric Deep Learning Blueprint

Bronstein et al. (2021) observe that a wide range of successful architectures are instances of a single template. A GDL model is constructed by composing:

1. **Equivariant layers** $f : \mathcal{X}(\Omega) \to \mathcal{X}(\Omega')$ that commute with the group action, $f \circ \rho(\mathfrak{g}) = \rho'(\mathfrak{g}) \circ f$.
2. **Local pooling (coarsening)** that progressively reduces the domain while respecting symmetry.
3. A final **invariant global pooling** producing a $\mathfrak{G}$-invariant representation for the task head.

Specialising the domain $\Omega$ and group $\mathfrak{G}$ recovers familiar models:

| Domain $\Omega$ | Group $\mathfrak{G}$ | Architecture |
|---|---|---|
| Grid $\mathbb{Z}^n$ | Translations | Convolutional Network (CNN) |
| Graph $G=(V,E)$ | Node permutations $S_n$ | Graph Neural Network (GNN) |
| Set | Permutations $S_n$ | Deep Sets / Transformer |
| Sphere $S^2$ | Rotations $SO(3)$ | Spherical CNN |
| Manifold $\mathcal{M}$ | Isometries / gauge | Gauge-equivariant CNN |

This unifying view is the through-line of the course: each subsequent week instantiates the blueprint on a new domain. Weeks 2&ndash;3 treat graphs and permutation symmetry; Week 4 treats 3D point clouds and $SE(3)$; Weeks 5&ndash;6 treat manifolds and the tools of Riemannian geometry.


## 6. Summary

- The **curse of dimensionality** makes unstructured high-dimensional learning intractable; **symmetry priors** are the remedy.
- A GDL problem is specified by a **domain** $\Omega$, a space of **signals** on it, and a **symmetry group** $\mathfrak{G}$ acting via a representation $\rho$.
- **Invariance** ($f \circ \rho(\mathfrak{g}) = f$) and **equivariance** ($f \circ \rho(\mathfrak{g}) = \sigma(\mathfrak{g}) \circ f$) are the design constraints we impose on layers.
- $SO(3)$ and $SE(3)$ are the Lie groups of rotations and rigid motions of 3D space.
- The **GDL blueprint** &mdash; equivariant layers, coarsening, invariant read-out &mdash; unifies CNNs, GNNs, transformers, and equivariant networks.

### References

- Bronstein, M. M., Bruna, J., Cohen, T., &amp; Veli&#269;kovi&#263;, P. (2021). *Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges.* arXiv:2104.13478.
- Hall, B. C. (2015). *Lie Groups, Lie Algebras, and Representations.* Springer.

### Exercises

1. Prove that the composition of two $\mathfrak{G}$-equivariant maps is $\mathfrak{G}$-equivariant. What is the representation on the output?
2. Show that global average pooling over a grid is translation-invariant.
3. Verify by hand that $SE(3)$ composition $(R_1,t_1)(R_2,t_2) = (R_1R_2,\,R_1 t_2 + t_1)$ satisfies the group associativity axiom.
